<a href="https://colab.research.google.com/github/rebecca-jf/PhD-project-code/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Training YOLO Models for Bumblebee Assays - Notebook structure

## 0. Imports

## 1. Extract frames from videos

## 2. Annotate frames with LabelMe

## 3. Convert LabelMe annotations to YOLO format


# =====================================
# WORKFLOW A
# Fine-tune an existing model
# =====================================

## A1. Separate already-seen and never-seen images

## A2. Create train / validation / test split

## A3. Create data.yaml

## A4. Evaluate existing model on untouched test set

## A5. Fine-tune existing model

## A6. Evaluate fine-tuned model

## A7. Compare models


# =====================================
# WORKFLOW B
# Train a new assay-specific model
# =====================================

## B1. Create train / validation / test split

## B2. Create data.yaml

## B3. Load official YOLO baseline

## B4. Train model

## B5. Evaluate model

## B6. Run model on full videos

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

from pathlib import Path

import cv2
import glob
import json
import random
import shutil

import numpy as np
import pandas as pd

import torch

from ultralytics import YOLO

In [ ]:
# ============================================================
# STEP 1: EXTRACT FRAMES FROM VIDEOS
# ============================================================
# Divides each video into equal temporal sections and randomly
# selects one frame from each section for annotation.


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

video_folder = Path("/path/to/videos")
output_folder = Path("/path/to/extracted_frames")

frames_per_video = 20

video_extensions = {".mp4", ".mov", ".avi", ".mkv"}

# Makes random frame selection reproducible
random.seed(42)


# ------------------------------------------------------------
# CREATE OUTPUT FOLDER
# ------------------------------------------------------------

output_folder.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# FIND VIDEOS
# ------------------------------------------------------------

video_files = [
    file for file in video_folder.iterdir()
    if file.suffix.lower() in video_extensions
]

print(f"Found {len(video_files)} videos.")


# ------------------------------------------------------------
# EXTRACT FRAMES
# ------------------------------------------------------------

for video_path in video_files:

    cap = cv2.VideoCapture(str(video_path))

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames <= 0:
        print(f"Could not read {video_path.name}")
        cap.release()
        continue

    print(
        f"\n{video_path.name}: "
        f"{total_frames} total frames"
    )

    # Divide video into equal temporal sections
    section_size = total_frames / frames_per_video

    selected_frames = []

    for i in range(frames_per_video):

        section_start = int(i * section_size)
        section_end = int((i + 1) * section_size) - 1

        section_end = min(
            section_end,
            total_frames - 1
        )

        # Select one random frame from this section
        frame_number = random.randint(
            section_start,
            section_end
        )

        selected_frames.append(frame_number)


    # --------------------------------------------------------
    # SAVE SELECTED FRAMES
    # --------------------------------------------------------

    saved_frames = 0

    for frame_number in selected_frames:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_number
        )

        success, frame = cap.read()

        if not success:
            print(
                f"Could not read frame {frame_number}"
            )
            continue

        output_name = (
            f"{video_path.stem}"
            f"_frame_{frame_number:06d}.jpg"
        )

        output_path = output_folder / output_name

        cv2.imwrite(
            str(output_path),
            frame
        )

        saved_frames += 1

    cap.release()

    print(f"Saved {saved_frames} frames.")


print("\nDone!")

Found 19 videos.

bee_0415_bee6_hand_color_opto_2026-04-16_00-26-06.mp4: 780 total frames
Saved 20 frames.

bee_0415_bee3_hand_color_opto_2026-04-15_21-10-45.mp4: 765 total frames
Saved 20 frames.

bee_0409_bee5_hand_color_opto_2026-04-09_13-08-45.mp4: 620 total frames
Saved 20 frames.

bee_0415_bee2_hand_color_opto_2026-04-15_19-22-22.mp4: 600 total frames
Saved 20 frames.

bee_0415_bee1_hand_color_opto_2026-04-15_16-17-34.mp4: 692 total frames
Could not read frame 688
Saved 20 frames.

bee_0415_bee2_hand_color_opto_2026-04-15_19-18-22.mp4: 729 total frames
Saved 20 frames.

bee_0409_bee6_hand_color_opto_2026-04-09_11-51-16.mp4: 585 total frames
Saved 20 frames.

bee_0415_bee6_hand_color_opto_2026-04-16_00-30-00.mp4: 670 total frames
Could not read frame 654
Saved 20 frames.

bee_0415_bee5_hand_color_opto_2026-04-15_23-22-51.mp4: 724 total frames
Saved 20 frames.

bee_0409_bee5_hand_color_opto_2026-04-09_13-03-08.mp4: 578 total frames
Saved 20 frames.

bee_0415_bee4_hand_color_opto_20

## 2. Annotate frames with LabelMe

Open the extracted frames in LabelMe and annotate each bee using a rectangular bounding box.

Use the class label:

`bee`

LabelMe creates one `.json` annotation file for each annotated image.

The next step converts these LabelMe annotations into YOLO format.

In [ ]:
# ============================================================
# STEP 3: CONVERT LABELME JSON ANNOTATIONS TO YOLO FORMAT
# ============================================================
# Converts rectangular LabelMe annotations into YOLO detection
# labels:
#
# class_id  x_center  y_center  width  height
#
# All coordinates are normalized to values between 0 and 1.


# ------------------------------------------------------------
# SETTINGS (adjust as needed)
# ------------------------------------------------------------

annotation_folder = Path("/path/to/annotated_frames")


# ------------------------------------------------------------
# FIND LABELME JSON FILES
# ------------------------------------------------------------

json_files = list(annotation_folder.glob("*.json"))

print(f"Found {len(json_files)} annotation files.")


# ------------------------------------------------------------
# CONVERT ANNOTATIONS
# ------------------------------------------------------------

for json_path in json_files:

    with open(json_path, "r") as f:
        data = json.load(f)

    image_height = data["imageHeight"]
    image_width = data["imageWidth"]

    yolo_lines = []

    for shape in data["shapes"]:

        # Only process rectangular bounding boxes
        if shape["shape_type"] != "rectangle":
            continue

        x1 = min(
            shape["points"][0][0],
            shape["points"][1][0]
        )

        x2 = max(
            shape["points"][0][0],
            shape["points"][1][0]
        )

        y1 = min(
            shape["points"][0][1],
            shape["points"][1][1]
        )

        y2 = max(
            shape["points"][0][1],
            shape["points"][1][1]
        )

        # Convert bounding box to YOLO format
        x_center = ((x1 + x2) / 2) / image_width
        y_center = ((y1 + y2) / 2) / image_height
        box_width = (x2 - x1) / image_width
        box_height = (y2 - y1) / image_height

        # Class 0 = bee
        yolo_lines.append(
            f"0 {x_center:.6f} {y_center:.6f} "
            f"{box_width:.6f} {box_height:.6f}"
        )

    # Save YOLO label with same filename as image/JSON
    txt_path = json_path.with_suffix(".txt")

    with open(txt_path, "w") as f:
        f.write("\n".join(yolo_lines))

print("Conversion complete!")

Conversion complete!


# Workflow A — Fine-tune an existing model

In [ ]:
# ============================================================
# WORKFLOW A: CREATE TRAIN / VALIDATION / TEST SPLIT
# FOR FINE-TUNING AN EXISTING MODEL
# ============================================================
#
# IMPORTANT:
# Images that were already used to train the existing model
# must NOT be placed in the validation or test set.
#
# Therefore:
# - all already-seen images -> training set
# - never-seen images -> split into train / validation / test


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

already_seen_folder = Path("/path/to/already_seen")
never_seen_folder = Path("/path/to/never_seen")

dataset_folder = Path("/path/to/YOLO_ready_dataset")

random_seed = 42

# Fractions applied only to the never-seen images
train_fraction = 0.70
val_fraction = 0.15
test_fraction = 0.15

image_extensions = {".jpg", ".jpeg", ".png"}


# ------------------------------------------------------------
# CREATE OUTPUT FOLDERS
# ------------------------------------------------------------

folders = {
    "train_images": dataset_folder / "images" / "train",
    "val_images": dataset_folder / "images" / "val",
    "test_images": dataset_folder / "images" / "test",
    "train_labels": dataset_folder / "labels" / "train",
    "val_labels": dataset_folder / "labels" / "val",
    "test_labels": dataset_folder / "labels" / "test",
}

for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# FIND IMAGES
# ------------------------------------------------------------

already_seen_images = sorted([
    file for file in already_seen_folder.iterdir()
    if file.suffix.lower() in image_extensions
])

never_seen_images = sorted([
    file for file in never_seen_folder.iterdir()
    if file.suffix.lower() in image_extensions
])

print(f"Already-seen images: {len(already_seen_images)}")
print(f"Never-seen images:   {len(never_seen_images)}")


# ------------------------------------------------------------
# SPLIT NEVER-SEEN IMAGES
# ------------------------------------------------------------

random.seed(random_seed)
random.shuffle(never_seen_images)

n_total = len(never_seen_images)

n_train = int(n_total * train_fraction)
n_val = int(n_total * val_fraction)

train_new = never_seen_images[:n_train]
val_new = never_seen_images[n_train:n_train + n_val]
test_new = never_seen_images[n_train + n_val:]


# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def copy_image_and_label(
    image_path,
    image_destination,
    label_destination
):

    label_path = image_path.with_suffix(".txt")

    if not label_path.exists():
        raise FileNotFoundError(
            f"No YOLO label found for {image_path.name}"
        )

    shutil.copy2(
        image_path,
        image_destination / image_path.name
    )

    shutil.copy2(
        label_path,
        label_destination / label_path.name
    )


# ------------------------------------------------------------
# ALREADY-SEEN IMAGES -> TRAIN
# ------------------------------------------------------------

for image_path in already_seen_images:
    copy_image_and_label(
        image_path,
        folders["train_images"],
        folders["train_labels"]
    )


# ------------------------------------------------------------
# NEVER-SEEN TRAIN IMAGES -> TRAIN
# ------------------------------------------------------------

for image_path in train_new:
    copy_image_and_label(
        image_path,
        folders["train_images"],
        folders["train_labels"]
    )


# ------------------------------------------------------------
# NEVER-SEEN VALIDATION IMAGES -> VALIDATION
# ------------------------------------------------------------

for image_path in val_new:
    copy_image_and_label(
        image_path,
        folders["val_images"],
        folders["val_labels"]
    )


# ------------------------------------------------------------
# NEVER-SEEN TEST IMAGES -> TEST
# ------------------------------------------------------------

for image_path in test_new:
    copy_image_and_label(
        image_path,
        folders["test_images"],
        folders["test_labels"]
    )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\nDataset created!")
print(f"Train:      {len(already_seen_images) + len(train_new)}")
print(f"Validation: {len(val_new)}")
print(f"Test:       {len(test_new)}")

OSError: [Errno 30] Read-only file system: '/path'

In [ ]:
f# ============================================================
# WORKFLOW A3: CREATE YOLO DATASET CONFIGURATION
# ============================================================
# Creates the data.yaml file that tells YOLO where to find
# the training, validation, and test images.

yaml_path = dataset_folder / "data.yaml"

yaml_content = f"""path: {dataset_folder}

train: images/train
val: images/val
test: images/test

names:
  0: bee
"""

yaml_path.write_text(yaml_content)

print(f"Created: {yaml_path}\n")
print(yaml_path.read_text())

path: /Users/rebeccafrei/Documents/jpegs_ymaze_color_training/YOLO_ready_dataset

train: images/train
val: images/val
test: images/test

names:
  0: bee



In [ ]:
# ============================================================
# WORKFLOW A4: EVALUATE BASELINE MODEL
# ============================================================
# Evaluates the existing model on the independent test set
# BEFORE fine-tuning.
#
# The test images must never have been used to train the
# existing model.


# ------------------------------------------------------------
# LOAD EXISTING MODEL (adjust path as needed)
# ------------------------------------------------------------

existing_model_path = Path("/path/to/existing_model.pt")

baseline_model = YOLO(existing_model_path)


# ------------------------------------------------------------
# EVALUATE ON TEST SET
# ------------------------------------------------------------

baseline_results = baseline_model.val(
    data=str(yaml_path),
    split="test"
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\nBASELINE MODEL — TEST SET")
print("=========================")

print(f"Precision:  {baseline_results.box.mp:.4f}")
print(f"Recall:     {baseline_results.box.mr:.4f}")
print(f"mAP50:      {baseline_results.box.map50:.4f}")
print(f"mAP50-95:   {baseline_results.box.map:.4f}")

Ultralytics 8.4.136 🚀 Python-3.12.10 torch-2.13.0 CPU (Apple M5)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3586.3±406.8 MB/s, size: 2495.0 KB)
val: Scanning /Users/rebeccafrei/Documents/jpegs_ymaze_color_training/YOLO_ready_dataset/labels/test... 7 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7/7 1.6Kit/s 0.0s
val: New cache created: /Users/rebeccafrei/Documents/jpegs_ymaze_color_training/YOLO_ready_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1s/it 1.1s
                   all          7          7      0.488      0.276      0.373      0.129
Speed: 0.4ms preprocess, 128.2ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /Users/rebeccafrei/runs/detect/val-2
Baseline Color-Maze performance
--------------------------------
Precision: 0.4883
Recall:    0.2757
mAP50:     0.3728


### Hardware check

The following check is relevant for Macs with Apple Silicon.

If `MPS available` is `True`, YOLO training can use the Apple GPU with:

`device="mps"`

On other systems, select the appropriate device (e.g. `"cuda"` for a compatible NVIDIA GPU or `"cpu"`).

In [ ]:
# ============================================================
# CHECK APPLE SILICON GPU AVAILABILITY
# ============================================================
# MPS allows PyTorch / YOLO to use the Apple Silicon GPU
# instead of running the training entirely on the CPU.

print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

In [ ]:
# ============================================================
# WORKFLOW A5: FINE-TUNE EXISTING MODEL
# ============================================================
# Continues training from an existing YOLO model using the
# prepared train/validation dataset.


# ------------------------------------------------------------
# LOAD EXISTING MODEL
# ------------------------------------------------------------

existing_model_path = Path("/path/to/existing_model.pt")

fine_tuning_model = YOLO(existing_model_path)


# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

training_results = fine_tuning_model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    device="mps",
    name="fine_tuned_model"
)

New https://pypi.org/project/ultralytics/8.4.137 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.136 🚀 Python-3.12.10 torch-2.13.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/rebeccafrei/Documents/jpegs_ymaze_color_training/YOLO_ready_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation.

       1/50      8.44G      1.617      1.121       1.42          7        640: 100% ━━━━━━━━━━━━ 12/12 2.5s/it 29.7s2.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.5s/it 2.5s
                   all          7          7       0.16      0.571      0.121     0.0414

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation.

       2/50      8.57G        1.7      1.312      1.524          7        640: 100% ━━━━━━━━━━━━ 12/12 1.6s/it 19.1s1.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.8it/s 0.6s
                   all          7          7          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

       9/50      8.45G      1.639      1.247      1.503          7        640: 100% ━━━━━━━━━━━━ 12/12 2.1s/it 25.8s1.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s
                   all          7          7     0.0328      0.286     0.0317     0.0095

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation.

      10/50      8.57G      1.622       1.13       1.43          3        640: 100% ━━━━━━━━━━━━ 12/12 2.1s/it 25.5s1.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.3it/s 0.8s
                   all          7          7      0.149      0.143     0.0375     0.0144

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation.

      11/50      8.45G      1.635      1.071      1.481          4        640: 100% ━━━━━━━━━━━━ 12/12 2.2s/it 26.2s1.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.7s
                   all          7          7    0.00495      0.143     0.0069    0.00276
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 1, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

11 epochs completed in 0.076 hours.
Optimizer stripped from /Users/rebeccafrei/runs/detect/color_maze_model/weights/last.pt, 40.5MB
Optimizer stripped from /Users/rebeccafrei/runs/detect/color_maze_model/weights/best.pt, 40.5MB

Validating /Users/rebeccafrei/runs/detect/color_maze_model/weights/best.pt...
Ultralytics 8.4.136 🚀 Python-3.12.10 torch-2.13.0 MPS (Apple M5)
YOLO11m summary

In [ ]:
# ============================================================
# WORKFLOW A6: EVALUATE FINE-TUNED MODEL
# ============================================================
# Loads the best weights produced during fine-tuning and
# evaluates them on the same independent test set that was
# used for the baseline evaluation.


# ------------------------------------------------------------
# LOAD BEST FINE-TUNED MODEL
# ------------------------------------------------------------

fine_tuned_model_path = Path(
    "/path/to/runs/detect/fine_tuned_model/weights/best.pt"
)

fine_tuned_model = YOLO(fine_tuned_model_path)


# ------------------------------------------------------------
# EVALUATE ON TEST SET
# ------------------------------------------------------------

fine_tuned_results = fine_tuned_model.val(
    data=str(yaml_path),
    split="test"
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\nFINE-TUNED MODEL — TEST SET")
print("===========================")

print(f"Precision:  {fine_tuned_results.box.mp:.4f}")
print(f"Recall:     {fine_tuned_results.box.mr:.4f}")
print(f"mAP50:      {fine_tuned_results.box.map50:.4f}")
print(f"mAP50-95:   {fine_tuned_results.box.map:.4f}")

Ultralytics 8.4.136 🚀 Python-3.12.10 torch-2.13.0 CPU (Apple M5)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.1 ms, read: 4269.4±224.2 MB/s, size: 2498.1 KB)
val: Scanning /Users/rebeccafrei/Documents/jpegs_ymaze_color_training/YOLO_ready_dataset/labels/test.cache... 7 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7/7 290.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.0s/it 1.0s
                   all          7          7      0.232          1      0.382      0.183
Speed: 0.3ms preprocess, 114.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /Users/rebeccafrei/runs/detect/val
Color-Maze model performance
--------------------------------
Precision: 0.2316
Recall:    1.0000
mAP50:     0.3825
mAP50-95:  0.1827


In [ ]:
# ============================================================
# WORKFLOW A7: COMPARE BASELINE AND FINE-TUNED MODEL
# ============================================================
# Compares both models on the SAME independent test set.


comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Fine-tuned"
    ],
    "Precision": [
        baseline_results.box.mp,
        fine_tuned_results.box.mp
    ],
    "Recall": [
        baseline_results.box.mr,
        fine_tuned_results.box.mr
    ],
    "mAP50": [
        baseline_results.box.map50,
        fine_tuned_results.box.map50
    ],
    "mAP50-95": [
        baseline_results.box.map,
        fine_tuned_results.box.map
    ]
})

comparison = comparison.round(4)

print(comparison.to_string(index=False))

In [ ]:
# ============================================================
# WORKFLOW A8: VISUAL MODEL COMPARISON ON A FULL VIDEO
# ============================================================
# Runs both the baseline and fine-tuned models on the same
# video using the same confidence threshold.


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

baseline_model_path = Path(
    "/path/to/existing_model.pt"
)

fine_tuned_model_path = Path(
    "/path/to/fine_tuned_model/weights/best.pt"
)

video_path = Path(
    "/path/to/test_video.mov"
)

confidence_threshold = 0.25


# ------------------------------------------------------------
# LOAD MODELS
# ------------------------------------------------------------

baseline_model = YOLO(baseline_model_path)
fine_tuned_model = YOLO(fine_tuned_model_path)


# ------------------------------------------------------------
# BASELINE MODEL
# ------------------------------------------------------------

baseline_model.predict(
    source=str(video_path),
    conf=confidence_threshold,
    save=True,
    show_conf=True,
    show_labels=True,
    device="mps",
    verbose=False,
    name="baseline_model_video"
)


# ------------------------------------------------------------
# FINE-TUNED MODEL
# ------------------------------------------------------------

fine_tuned_model.predict(
    source=str(video_path),
    conf=confidence_threshold,
    save=True,
    show_conf=True,
    show_labels=True,
    device="mps",
    verbose=False,
    name="fine_tuned_model_video"
)


print("Video comparison complete!")

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



: 

Ab hier dann workflow B

In [ ]:
# ============================================================
# WORKFLOW B: CREATE TRAIN / VALIDATION / TEST SPLIT
# FOR A NEW ASSAY-SPECIFIC MODEL
# ============================================================
#
# Use this workflow when training a new model from a fresh
# pretrained YOLO baseline (e.g. yolo11m.pt).
#
# All annotated images are randomly split into:
# - training set
# - validation set
# - test set


# ------------------------------------------------------------
# SETTINGS (adjust paths and fractions as needed)
# ------------------------------------------------------------

annotated_folder = Path("/path/to/annotated_frames")

dataset_folder = Path("/path/to/YOLO_ready_dataset")

random_seed = 42

train_fraction = 0.70
val_fraction = 0.15
test_fraction = 0.15

image_extensions = {".jpg", ".jpeg", ".png"}


# ------------------------------------------------------------
# CREATE OUTPUT FOLDERS
# ------------------------------------------------------------

folders = {
    "train_images": dataset_folder / "images" / "train",
    "val_images": dataset_folder / "images" / "val",
    "test_images": dataset_folder / "images" / "test",
    "train_labels": dataset_folder / "labels" / "train",
    "val_labels": dataset_folder / "labels" / "val",
    "test_labels": dataset_folder / "labels" / "test",
}

for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# FIND ANNOTATED IMAGES
# ------------------------------------------------------------

all_images = sorted([
    file for file in annotated_folder.iterdir()
    if file.suffix.lower() in image_extensions
])

print(f"Found {len(all_images)} annotated images.")


# ------------------------------------------------------------
# SHUFFLE AND SPLIT
# ------------------------------------------------------------

random.seed(random_seed)
random.shuffle(all_images)

n_total = len(all_images)

n_train = int(n_total * train_fraction)
n_val = int(n_total * val_fraction)

train_images = all_images[:n_train]
val_images = all_images[n_train:n_train + n_val]
test_images = all_images[n_train + n_val:]


# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def copy_image_and_label(
    image_path,
    image_destination,
    label_destination
):

    label_path = image_path.with_suffix(".txt")

    if not label_path.exists():
        raise FileNotFoundError(
            f"No YOLO label found for {image_path.name}"
        )

    shutil.copy2(
        image_path,
        image_destination / image_path.name
    )

    shutil.copy2(
        label_path,
        label_destination / label_path.name
    )


# ------------------------------------------------------------
# COPY TRAINING DATA
# ------------------------------------------------------------

for image_path in train_images:
    copy_image_and_label(
        image_path,
        folders["train_images"],
        folders["train_labels"]
    )


# ------------------------------------------------------------
# COPY VALIDATION DATA
# ------------------------------------------------------------

for image_path in val_images:
    copy_image_and_label(
        image_path,
        folders["val_images"],
        folders["val_labels"]
    )


# ------------------------------------------------------------
# COPY TEST DATA
# ------------------------------------------------------------

for image_path in test_images:
    copy_image_and_label(
        image_path,
        folders["test_images"],
        folders["test_labels"]
    )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\nDataset created!")
print(f"Train:      {len(train_images)}")
print(f"Validation: {len(val_images)}")
print(f"Test:       {len(test_images)}")

In [ ]:
# ============================================================
# WORKFLOW B2: CREATE YOLO DATASET CONFIGURATION
# ============================================================
# Creates the data.yaml file for the new assay-specific dataset.


yaml_path = dataset_folder / "data.yaml"

yaml_content = f"""path: {dataset_folder}

train: images/train
val: images/val
test: images/test

names:
  0: bee
"""

yaml_path.write_text(yaml_content)

print(f"Created: {yaml_path}\n")
print(yaml_path.read_text())

In [ ]:
# ============================================================
# WORKFLOW B3: LOAD FRESH PRETRAINED YOLO BASELINE
# ============================================================
# Starts from the official pretrained YOLO11m weights instead
# of from a previously fine-tuned assay model.


new_model = YOLO("yolo11m.pt")

print("Fresh YOLO11m baseline loaded.")

In [ ]:
# ============================================================
# WORKFLOW B4: TRAIN NEW ASSAY-SPECIFIC MODEL
# ============================================================
# Fine-tunes the fresh pretrained YOLO11m baseline on the
# new assay-specific dataset.


training_results = new_model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    device="mps",
    name="assay_specific_model"
)

### Training settings

- `epochs=50`: maximum number of training epochs
- `imgsz=640`: input image size
- `batch=8`: number of images processed per batch
- `patience=10`: early stopping if validation performance does not improve for 10 epochs
- `device="mps"`: use Apple Silicon GPU
- `name`: name of the training run saved under `runs/detect/`

In [ ]:
# ============================================================
# WORKFLOW B5: EVALUATE TRAINED MODEL
# ============================================================
# Loads the best weights from training and evaluates them
# on the independent test set.


# ------------------------------------------------------------
# LOAD BEST TRAINED MODEL
# ------------------------------------------------------------

trained_model_path = Path(
    "/path/to/runs/detect/assay_specific_model/weights/best.pt"
)

trained_model = YOLO(trained_model_path)


# ------------------------------------------------------------
# EVALUATE ON TEST SET
# ------------------------------------------------------------

test_results = trained_model.val(
    data=str(yaml_path),
    split="test"
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\nASSAY-SPECIFIC MODEL — TEST SET")
print("===============================")

print(f"Precision:  {test_results.box.mp:.4f}")
print(f"Recall:     {test_results.box.mr:.4f}")
print(f"mAP50:      {test_results.box.map50:.4f}")
print(f"mAP50-95:   {test_results.box.map:.4f}")

In [ ]:
# ============================================================
# WORKFLOW B6: RUN TRAINED MODEL ON A FULL VIDEO
# ============================================================
# Applies the trained assay-specific model to a representative
# full-length video for visual inspection.


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

video_path = Path(
    "/path/to/representative_video.mov"
)

confidence_threshold = 0.25


# ------------------------------------------------------------
# RUN VIDEO PREDICTION
# ------------------------------------------------------------

video_results = trained_model.predict(
    source=str(video_path),
    conf=confidence_threshold,
    save=True,
    show_conf=True,
    show_labels=True,
    device="mps",
    verbose=False,
    name="assay_specific_video_test"
)

print("Video prediction complete!")

### Visual inspection on full videos

Before using detections for behavioral analysis, run the trained model on
representative full-length videos.

Check for:

- missed bees
- false-positive detections
- unstable bounding boxes
- systematic errors caused by the apparatus, background, reflections, or image edges
- whether the chosen confidence threshold is appropriate

If videos will be cropped before analysis, evaluate the model on videos
processed in the same way.